**Record of a July 2026 Colab run.** This notebook keeps the original outputs, though its first cell was replaced to clone this repository, so those outputs came from the code of the time. To reproduce the result today, use [`docs/reproducing.md`](../docs/reproducing.md).

# Phase 5 Gate A — paired checkpoint re-score

This notebook performs inference only. It compares the frozen LFM2-350M model and the existing seed-23 adapter under matched training, ablation, current-evaluation, and greedy decoders. Each stochastic condition uses the same 50 generation seeds for both policies. The result estimates pass@1 on one seed-invariant `click-test` prompt; it is not held-out task generalization.

## 1. Fetch the repository

In [ ]:
# Fetch the public repository.
import os
import subprocess
from pathlib import Path

repository_path = Path('/content/browser-control')
if not repository_path.exists():
    subprocess.run(['git', 'clone', '--branch', 'lfm-browser-control',
                    'https://github.com/DerrickKirimi/browser-control.git', str(repository_path)], check=True)
os.chdir(repository_path)


## 2. Install without replacing Colab's PyTorch

In [2]:
from importlib.metadata import version

torch_before = version('torch')
%pip install -q -e ".[environment,model,training]" pytest
torch_after = version('torch')
print('PyTorch:', torch_before, '->', torch_after)
if torch_after != torch_before:
    raise RuntimeError('Project installation replaced Colab PyTorch')
%pip uninstall -y torchao
!python -m playwright install --with-deps chromium

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.6/191.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.7/625.7 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

## 3. Validate the runtime and Gate A contracts

In [3]:
!python scripts/check_colab_gpu.py
!python -m pytest -q tests/test_evaluation.py tests/test_rescoring.py tests/test_transformers_policy.py

{
  "cudnn_version": 91900,
  "device": {
    "compute_capability": "7.5",
    "index": 0,
    "name": "Tesla T4",
    "supports_bfloat16": false,
    "total_memory_bytes": 15637086208,
    "training_dtype": "float16"
  },
  "torch_cuda_version": "12.8",
  "torch_version": "2.11.0+cu128"
}
................                                                         [100%]


## 4. Upload and extract the existing seed-23 adapter

This run uploaded the adapter archive from the seed-23 pilot, which is not published.

In [4]:
import shutil
from pathlib import Path

from google.colab import files

uploaded = files.upload()
archives = [Path(name) for name in uploaded if name.endswith('.zip')]
if len(archives) != 1:
    raise RuntimeError('Upload exactly one adapter ZIP')
adapter_dir = Path('artifacts/phase5/gate-a/uploaded-adapter')
if adapter_dir.exists():
    shutil.rmtree(adapter_dir)
adapter_dir.mkdir(parents=True)
shutil.unpack_archive(archives[0], adapter_dir)
weights = adapter_dir / 'adapter_model.safetensors'
if not weights.is_file():
    raise RuntimeError(f'Adapter archive lacks {weights.name}')
print('Adapter ready:', adapter_dir, weights.stat().st_size, 'bytes')

[browser download widget removed]

Saving phase5-click-test-seed-0023-steps-0060-adapter.zip to phase5-click-test-seed-0023-steps-0060-adapter.zip
Adapter ready: artifacts/phase5/gate-a/uploaded-adapter 10703800 bytes


## 5. Run paired frozen/trained re-evaluation

In [5]:
!python scripts/rescore_phase5_adapter.py --adapter-dir artifacts/phase5/gate-a/uploaded-adapter --samples 50 --first-seed 2000 --output artifacts/phase5/gate-a/phase5-gate-a-rescore.json

2026-07-21 21:48:33.827295: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
tokenizer_config.json: 91.6kB [00:00, 178MB/s]
tokenizer.json: 4.73MB [00:00, 84.7MB/s]
special_tokens_map.json: 100% 434/434 [00:00<00:00, 3.70MB/s]
chat_template.jinja: 1.38kB [00:00, 8.88MB/s]
config.json: 100% 999/999 [00:00<00:00, 9.25MB/s]
model.safetensors: 100% 709M/709M [00:03<00:00, 185MB/s]
generation_config.json: 100% 137/137 [00:00<00:00, 856kB/s]
{
  "conditions": {
    "current-evaluation": {
      "frozen": "0/50",
      "success_rate_delta": 0,
      "trained": "0/50"
    },
    "evaluation-no-repetition-penalty": {
      "frozen": "0/50",
      "success_rate_delta": 0.24,
      "trained": "12/50"
    },
    "greedy": {
      "frozen": "0/1",
      "succe

## 6. Inspect the comparison and download the report

In [6]:
import json
from pathlib import Path

from google.colab import files

report_path = Path('artifacts/phase5/gate-a/phase5-gate-a-rescore.json')
report = json.loads(report_path.read_text())
for decoder_name in report['frozen']:
    frozen = report['frozen'][decoder_name]
    trained = report['trained'][decoder_name]
    print(decoder_name)
    print('  frozen:', frozen['successes'], '/', frozen['samples'], frozen['wilson_95'])
    print('  trained:', trained['successes'], '/', trained['samples'], trained['wilson_95'])
files.download(str(report_path))

current-evaluation
  frozen: 0 / 50 [6.938893903907228e-18, 0.07134759913335872]
  trained: 0 / 50 [6.938893903907228e-18, 0.07134759913335872]
evaluation-no-repetition-penalty
  frozen: 0 / 50 [6.938893903907228e-18, 0.07134759913335872]
  trained: 12 / 50 [0.1429739139699173, 0.3741268375794292]
greedy
  frozen: 0 / 1 None
  trained: 0 / 1 None
training
  frozen: 2 / 50 [0.011038884327619805, 0.1346009068750702]
  trained: 16 / 50 [0.20758216260656845, 0.45810297308144077]


[browser download widget removed]

[browser download widget removed]